# Short Notebook 2 - Two-Stage XGBoost Model

**Team:** [35] ∫ √(tan x) dx

**Members:** Antonije Mirkovic (115014), ...

---

This notebook implements a two-stage forecasting approach:
1. **Stage 1:** XGBoost classifier predicts order placement probability for 2025
2. **Stage 2:** Historical delivery rates are multiplied by order probability to generate final predictions

## Imports and Configuration

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix, accuracy_score

# Global shrink factor for predictions
GLOBAL_SHRINK = 1.0

## Load Data

In [2]:
receivals = pd.read_csv('./Project_materials/data/kernel/receivals.csv')
purchase_orders = pd.read_csv('./Project_materials/data/kernel/purchase_orders.csv')
prediction_mapping = pd.read_csv('./Project_materials/data/prediction_mapping.csv')
sample_submission = pd.read_csv('./Project_materials/data/sample_submission.csv')

# Convert dates
receivals['date_arrival'] = pd.to_datetime(receivals['date_arrival'], utc=True).dt.tz_localize(None)
purchase_orders['delivery_date'] = pd.to_datetime(purchase_orders['delivery_date'], utc=True).dt.tz_localize(None)
purchase_orders['created_date_time'] = pd.to_datetime(purchase_orders['created_date_time'], utc=True).dt.tz_localize(None)
prediction_mapping['forecast_start_date'] = pd.to_datetime(prediction_mapping['forecast_start_date'])
prediction_mapping['forecast_end_date'] = pd.to_datetime(prediction_mapping['forecast_end_date'])

print(f"Receivals: {len(receivals):,} rows")
print(f"Purchase orders: {len(purchase_orders):,} rows")
print(f"Predictions to make: {len(prediction_mapping):,}")

Receivals: 122,590 rows
Purchase orders: 33,171 rows
Predictions to make: 30,450


## Data Preprocessing

In [3]:
# Deduplicate purchase orders (keep first by creation time)
purchase_orders_dedup = purchase_orders.sort_values('created_date_time').drop_duplicates('purchase_order_id', keep='first')
print(f"Purchase orders: {len(purchase_orders):,} → {len(purchase_orders_dedup):,} unique orders")

# Merge receivals with purchase order creation times
receivals_merged = receivals.merge(
    purchase_orders_dedup[['purchase_order_id', 'created_date_time']], 
    on='purchase_order_id', 
    how='left'
)

# Clean data
receivals_merged = receivals_merged[receivals_merged['net_weight'] > 0]
receivals_merged = receivals_merged[receivals_merged['rm_id'].notna()]
receivals_merged = receivals_merged.sort_values('date_arrival')
receivals_merged['year'] = receivals_merged['date_arrival'].dt.year

receivals = receivals[receivals['net_weight'] > 0]
receivals = receivals[receivals['rm_id'].notna()]
receivals = receivals.sort_values('date_arrival')

print(f"\nClean receivals: {len(receivals_merged):,}")
print(f"Materials: {receivals_merged['rm_id'].nunique()}")
print(f"Date range: {receivals_merged['date_arrival'].min().date()} to {receivals_merged['date_arrival'].max().date()}")

Purchase orders: 33,171 → 8,135 unique orders

Clean receivals: 122,383
Materials: 203
Date range: 2004-06-15 to 2024-12-19


## Define Active Years Based on Order Placement

In [4]:
# Track which years each material had orders placed
df_with_orders = receivals_merged[receivals_merged['created_date_time'].notna()].copy()
df_with_orders['order_year'] = df_with_orders['created_date_time'].dt.year
material_years = df_with_orders.groupby('rm_id')['order_year'].apply(
    lambda x: set(x.unique())
).to_dict()

print(f"Materials with order history: {len(material_years)}")

# Identify materials by history
materials_2023 = receivals_merged[receivals_merged['year'] <= 2023]['rm_id'].unique()
materials_2024_new = receivals_merged[receivals_merged['year'] == 2024]['rm_id'].unique()
materials_2024_new = [m for m in materials_2024_new if m not in materials_2023]

print(f"Materials with history through 2023: {len(materials_2023)}")
print(f"New materials in 2024: {len(materials_2024_new)}")

Materials with order history: 203
Materials with history through 2023: 188
New materials in 2024: 15


## Feature Engineering Function

Creates 8 features from 1-year historical window for order prediction.

In [5]:
def create_order_features(rm_id, prediction_year, df_all, material_years):
    """
    Create features for order prediction using 1-year lookback window.
    Uses deliveries and purchase order times from (prediction_year - 1).
    """
    # Get data from last year only
    window_start_year = prediction_year - 1
    window_end_year = prediction_year - 1
    
    hist_data = df_all[
        (df_all['rm_id'] == rm_id) & 
        (df_all['year'] >= window_start_year) &
        (df_all['year'] <= window_end_year)
    ].copy()
    
    if len(hist_data) == 0:
        return None
    
    cutoff_date = pd.Timestamp(f'{prediction_year-1}-12-31')
    hist_data_cutoff = hist_data[hist_data['date_arrival'] <= cutoff_date]
    
    if len(hist_data_cutoff) == 0:
        return None
    
    features = {
        'rm_id': rm_id,
        'prediction_year': prediction_year,
    }
    
    # Target: Were orders placed in prediction_year?
    active_years = material_years.get(rm_id, set())
    features['target'] = 1 if prediction_year in active_years else 0
    
    # Feature 1: Days since last delivery
    last_delivery_date = hist_data_cutoff['date_arrival'].max()
    features['days_since_last_delivery'] = (cutoff_date - last_delivery_date).days
    
    # Feature 2: Days since last purchase order
    last_order_date = hist_data_cutoff['created_date_time'].max()
    if pd.notna(last_order_date):
        features['days_since_last_purchase_order'] = (cutoff_date - last_order_date).days
    else:
        features['days_since_last_purchase_order'] = np.nan
    
    # Feature 3: Weight in last 3 months
    three_months_ago = cutoff_date - timedelta(days=90)
    recent_data = hist_data_cutoff[hist_data_cutoff['date_arrival'] > three_months_ago]
    features['weight_last_3_months'] = recent_data['net_weight'].sum()
    
    # Feature 4: Last 3m weight vs historical average 3m weight
    historical_3m_weights = []
    for y in hist_data_cutoff['year'].unique():
        year_data = hist_data_cutoff[hist_data_cutoff['year'] == y]
        for quarter_start_month in [1, 4, 7, 10]:
            quarter_data = year_data[
                (year_data['date_arrival'].dt.month >= quarter_start_month) &
                (year_data['date_arrival'].dt.month < quarter_start_month + 3)
            ]
            if len(quarter_data) > 0:
                historical_3m_weights.append(quarter_data['net_weight'].sum())
    
    avg_historical_3m_weight = np.mean(historical_3m_weights) if len(historical_3m_weights) > 0 else 0
    
    if avg_historical_3m_weight > 0:
        features['weight_last_3m_vs_avg_historical_3m'] = features['weight_last_3_months'] / avg_historical_3m_weight
    else:
        features['weight_last_3m_vs_avg_historical_3m'] = 0
    
    # Feature 5: Number of deliveries in last 3 months
    features['num_deliveries_last_3_months'] = len(recent_data)
    
    # Feature 6: Number of orders in last 3 months
    orders_last_3m = hist_data_cutoff[
        (hist_data_cutoff['created_date_time'].notna()) &
        (hist_data_cutoff['created_date_time'] > three_months_ago)
    ]
    features['num_orders_last_3_months'] = orders_last_3m['purchase_order_id'].nunique()
    
    # Feature 7: Average days between deliveries in last year
    twelve_months_ago = cutoff_date - timedelta(days=365)
    deliveries_last_year = hist_data_cutoff[hist_data_cutoff['date_arrival'] > twelve_months_ago].copy()
    deliveries_last_year = deliveries_last_year.sort_values('date_arrival')
    
    if len(deliveries_last_year) > 1:
        gaps_last_year = deliveries_last_year['date_arrival'].diff().dt.days.dropna()
        features['avg_days_between_deliveries_last_year'] = gaps_last_year.mean()
    elif len(deliveries_last_year) == 1:
        features['avg_days_between_deliveries_last_year'] = 365
    else:
        features['avg_days_between_deliveries_last_year'] = 365
    
    # Feature 8: Average gap including year edges
    year = prediction_year - 1
    year_start = pd.Timestamp(f"{year}-01-01")
    year_end = pd.Timestamp(f"{year}-12-31")
    deliveries = hist_data_cutoff.sort_values("date_arrival")["date_arrival"]
    
    if len(deliveries) > 0:
        full_timeline = [year_start] + list(deliveries) + [year_end]
        gaps_incl_edges = pd.Series(full_timeline).diff().dt.days.dropna()
        features["avg_gap_including_edges"] = gaps_incl_edges.mean()
    else:
        features["avg_gap_including_edges"] = 365
    
    return features

## Create Training Data

Build multi-year training dataset for order prediction.

In [6]:
training_data = []
max_train_year = 2024

for rm_id, active_years in material_years.items():
    first_year = min(active_years)
    for prediction_year in range(first_year + 1, max_train_year + 1):
        # Require data in previous year
        has_prev_year_data = (
            (receivals_merged['rm_id'] == rm_id) &
            (receivals_merged['year'] == prediction_year - 1)
        ).any()
        if not has_prev_year_data:
            continue

        features = create_order_features(rm_id, prediction_year, receivals_merged, material_years)
        if features is not None:
            training_data.append(features)

df_train = pd.DataFrame(training_data)
print(f"Training samples: {len(df_train):,}")

feature_cols = [
    'weight_last_3m_vs_avg_historical_3m',
    'days_since_last_delivery',
    'avg_days_between_deliveries_last_year',
    'weight_last_3_months',
    'num_deliveries_last_3_months',
    'num_orders_last_3_months',
    'days_since_last_purchase_order',
    'avg_gap_including_edges'
]

X = df_train[feature_cols]
y = df_train['target']

print(f"\nTarget distribution:")
print(f"  Orders placed (1): {y.sum()} ({100*y.mean():.1f}%)")
print(f"  No orders (0): {(y==0).sum()} ({100*(1-y.mean()):.1f}%)")

Training samples: 685

Target distribution:
  Orders placed (1): 510 (74.5%)
  No orders (0): 175 (25.5%)


## Train Order Prediction Model

XGBoost classifier with grid search for hyperparameter tuning.

In [7]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_base = XGBClassifier(random_state=42, eval_metric='logloss')
grid_search = GridSearchCV(xgb_base, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=0)
grid_search.fit(X, y)

order_model = grid_search.best_estimator_
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")

# In-sample validation
y_pred = order_model.predict(X)
accuracy = accuracy_score(y, y_pred)
cm = confusion_matrix(y, y_pred)
tn, fp, fn, tp = cm.ravel()

print(f"\nValidation Accuracy: {accuracy:.4f}")
print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

Best parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'subsample': 1.0}
Best CV score: 0.8744

Validation Accuracy: 0.9095
Confusion Matrix: TN=144, FP=31, FN=31, TP=479
Precision: 0.9392, Recall: 0.9392, F1: 0.9392


## Generate 2025 Order Predictions

In [8]:
prediction_date_2025 = pd.Timestamp('2025-01-01')
all_materials = prediction_mapping['rm_id'].unique()
order_predictions_2025 = {}

for rm_id in all_materials:
    if rm_id in materials_2023 or rm_id in materials_2024_new:
        features = create_order_features(rm_id, 2025, receivals_merged, material_years)
        if features is not None:
            feat_vector = pd.DataFrame([features])[feature_cols]
            prob = order_model.predict_proba(feat_vector)[0, 1]
            order_predictions_2025[rm_id] = prob
        else:
            order_predictions_2025[rm_id] = 0.0
    else:
        order_predictions_2025[rm_id] = 0.0

print(f"Order predictions computed: {len(order_predictions_2025)}")
print(f"  Predicted active (prob > 0.5): {sum(1 for p in order_predictions_2025.values() if p > 0.5)}")
print(f"  Predicted uncertain (0.3-0.5): {sum(1 for p in order_predictions_2025.values() if 0.3 <= p <= 0.5)}")
print(f"  Predicted inactive (prob < 0.3): {sum(1 for p in order_predictions_2025.values() if p < 0.3)}")

Order predictions computed: 203
  Predicted active (prob > 0.5): 43
  Predicted uncertain (0.3-0.5): 9
  Predicted inactive (prob < 0.3): 151


## Compute Historical Delivery Rates

In [9]:
hist_through_2024 = receivals_merged[receivals_merged['date_arrival'] < prediction_date_2025]
historical_rates = {}

for rm_id in all_materials:
    rm_hist = hist_through_2024[hist_through_2024['rm_id'] == rm_id]
    
    if len(rm_hist) == 0:
        historical_rates[rm_id] = {
            'daily_rate_365d': 0.0,
            'daily_rate_180d': 0.0,
            'total_365d': 0.0,
            'days_since_last': 999
        }
        continue
    
    cutoff_365 = prediction_date_2025 - timedelta(days=365)
    recent_365 = rm_hist[rm_hist['date_arrival'] >= cutoff_365]
    
    cutoff_180 = prediction_date_2025 - timedelta(days=180)
    recent_180 = rm_hist[rm_hist['date_arrival'] >= cutoff_180]
    
    weight_365 = recent_365['net_weight'].sum()
    weight_180 = recent_180['net_weight'].sum()
    
    daily_rate_365 = weight_365 / 365.0
    daily_rate_180 = weight_180 / 180.0
    
    days_since = (prediction_date_2025 - rm_hist['date_arrival'].max()).days
    
    historical_rates[rm_id] = {
        'daily_rate_365d': daily_rate_365,
        'daily_rate_180d': daily_rate_180,
        'total_365d': weight_365,
        'days_since_last': days_since
    }

print(f"Historical rates computed for {len(historical_rates)} materials")

Historical rates computed for 203 materials


## Generate Final Predictions

Multiply historical rates by order probability, with probability floor for large active materials and guardrails for inactive materials.

In [10]:
predictions = []

for idx, row in prediction_mapping.iterrows():
    rm_id = row['rm_id']
    forecast_end = row['forecast_end_date']
    horizon = (forecast_end - prediction_date_2025).days + 1
    
    order_prob = order_predictions_2025.get(rm_id, 0.0)
    
    rates = historical_rates.get(rm_id, {
        'daily_rate_365d': 0.0,
        'daily_rate_180d': 0.0,
        'total_365d': 0.0,
        'days_since_last': 999
    })
    
    # Choose rate: prefer 180d if non-zero, else 365d
    if rates['daily_rate_180d'] > 0:
        daily_rate = rates['daily_rate_180d']
    else:
        daily_rate = rates['daily_rate_365d']
    
    base_pred = daily_rate * horizon
    
    total_365 = rates['total_365d']
    days_inactive = rates['days_since_last']
    
    # Probability floor for large, recently active materials
    effective_prob = order_prob
    if (total_365 >= 300_000) and (days_inactive <= 240):
        effective_prob = max(effective_prob, 0.6)
    
    final_pred = base_pred * effective_prob
    
    # Inactivity guardrail
    if days_inactive > 365:
        final_pred = final_pred * 0.2
    
    # Upper bound cap based on past 365 days
    if total_365 > 0:
        annual_total = total_365
        cap = (annual_total / 365.0) * horizon * 1.5
        final_pred = min(final_pred, cap)
    
    final_pred = max(0.0, final_pred)
    final_pred = GLOBAL_SHRINK * final_pred
    
    predictions.append({
        'ID': row['ID'],
        'predicted_weight': final_pred
    })

predictions_df = pd.DataFrame(predictions)

print(f"Total predictions: {len(predictions_df):,}")
print(f"Total predicted: {predictions_df['predicted_weight'].sum():,.0f} kg")
print(f"Mean: {predictions_df['predicted_weight'].mean():,.0f} kg")
print(f"Median: {predictions_df['predicted_weight'].median():,.0f} kg")
print(f"Non-zero: {(predictions_df['predicted_weight'] > 0).sum():,} "
      f"({100*(predictions_df['predicted_weight'] > 0).mean():.1f}%)")

Total predictions: 30,450
Total predicted: 2,338,684,692 kg
Mean: 76,804 kg
Median: 0 kg
Non-zero: 9,000 (29.6%)


## Save Submission

In [11]:
submission = sample_submission.copy()
submission['predicted_weight'] = predictions_df['predicted_weight'].values
submission.to_csv('Short_notebook_2.csv', index=False)

print("Submission saved: Short_notebook_2.csv")

Submission saved: Short_notebook_2.csv
